# Server setup + train (trần 15h, snapshot giờ 9)

Chạy từng cell theo thứ tự. Dừng ngay khi cell kiểm tra FAIL, không chạy train.
Không mount Google Drive. Cuối giờ copy `models/snapshot_9h.pt` và `models/best_model.pt` về máy.


In [ ]:
!nvidia-smi
!python --version
!pip --version

In [ ]:
!pip install -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

import torch

print('python:', sys.version.split()[0])
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for the 15h training run')
props = torch.cuda.get_device_properties(0)
print('gpu:', props.name, f'| vram: {props.total_memory / 1024 ** 3:.1f}GB')
print(f'cpu: {os.cpu_count()}')
engine = Path('pikafish/Pikafish-Linux-x86-64-universal')
if engine.is_file():
    engine.chmod(engine.stat().st_mode | 0o111)
required = [
    'pikafish/Pikafish-Linux-x86-64-universal',
    'pikafish/pikafish.nnue',
    'configs/train_depth2_16h.yaml',
    'scripts/run_16h.py',
    'scripts/train.py',
    'scripts/build_teacher_labels.py',
    'scripts/pretrain_supervised.py',
    'scripts/measure_vs_pikafish.py',
]
missing = [path for path in required if not Path(path).is_file()]
for path in required:
    print(('OK  ' if Path(path).is_file() else 'MISS'), path)
if missing:
    raise FileNotFoundError('Missing required files: ' + ', '.join(missing))


In [ ]:
!python tests/test_game.py
!python -m pytest tests/test_engine.py tests/test_search_score.py tests/test_teacher_label.py tests/test_worker_select.py tests/test_sparring_game.py -q

In [ ]:
# Trần 15h. Giờ 9: snapshot models/snapshot_9h.pt. Giờ 15: dừng hẳn.
!python scripts/run_16h.py --config configs/train_depth2_16h.yaml
